# Домашнее задание: Традиционный ML в Multi-Agent системах

**Задача:** реализовать маршрутизацию запросов банковского чат-бота, сравнить подходы - baseline (LLM) и proposed (ML).





## 0. Установка и импорт

In [47]:
import time
import numpy as np
import pandas as pd
import requests

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from google.colab import userdata

## 1. Данные

Синтетический датасет (сгенерирован DeepSeek) состоит из 96 запросов по 8 категориям: balance, history, payments, credit, cards, currency, deposit, complaint.

In [42]:
df = pd.read_csv('bank_messages.csv')
display(df.head())
print("\nРазмер датасета:", len(df))
print("\nРаспределение запросов по категориям:\n", df["category"].value_counts())

,text,category
0,Какой баланс на моей дебетовой карте?,balance
1,Покажите последние 5 операций по счёту,history
2,Переведите 1500 рублей на номер +7 912 345-67-89,payments
3,Хочу взять кредит наличными 300 000 рублей,credit
4,"Заблокируйте мою карту, я её потерял",cards



Размер датасета: 96

Распределение запросов по категориям:
 category
balance      12
history      12
payments     12
credit       12
cards        12
currency     12
deposit      12
complaint    12
Name: count, dtype: int64


In [56]:
# Деление на обучающую и тестовую выборки
train_df, test_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df['category'])
print(f"train: {len(train_df)}, test: {len(test_df)}")

train: 72, test: 24


## 2. Baseline

Для маршрутизации используется YandexGPT 5 Lite: модель должна вернуть только метку класса, основываясь на инструкции и тексте запроса. В промпте используем технику few-shot learning.

In [74]:
def classify_with_llm(text: str):
    """Совершает запросы к YandexGPT 5 Lite, возвращает текст ответа,
    время, затраченное на генерацию, количество входных и выходных токенов"""

    API_KEY = userdata.get('yandexgpt_api_key')
    CATALOG_ID = 'b1gqp5r9omav4q9ou3d5'

    system_prompt = """
    Ты — классификатор запросов клиентов банка. Твоя задача: определить категорию пользовательского обращения из заданного списка.

    Категории (всего 8):
    - balance — запросы об остатке денег на счёте, карте, вкладе, кредитном лимите, доступных средствах.
    - history — запросы о выписке, истории операций, транзакциях, чеках, движении средств за период.
    - payments — запросы о переводах, оплате услуг, автоплатежах, пополнении счёта, переводах по номеру телефона или реквизитам.
    - credit — запросы о кредитах: условиях, ставках, рефинансировании, досрочном погашении, одобрении, остатке по кредиту.
    - cards — запросы о картах: блокировка, разблокировка, перевыпуск, активация, смена ПИН-кода, добавление в Apple Pay/Google Pay, лимиты.
    - currency — запросы о курсах валют (рубль, доллар, евро, юань, лира и др.), обмене валюты, конвертации.
    - deposit — запросы о вкладах (депозитах): открытие, пополнение, частичное снятие, процентные ставки, капитализация, досрочное закрытие.
    - complaint — жалобы: на работу операторов, двойное списание, неработающее приложение, мошенничество, некорректные комиссии, банкоматы, спам.

    Важные правила:
    - Если запрос относится к нескольким категориям, выбери самую подходящую по основной сути.
    - Если запрос про остаток по кредиту → credit (а не balance).
    - Если запрос про курс валют и конвертацию → currency (а не payments).
    - Если запрос про историю переводов → history (а не payments).
    - Жалобы всегда помещай в complaint, даже если они содержат вопрос о балансе или операции.

    Выходной формат: ТОЛЬКО название категории на латинице (balance, history, payments, credit, cards, currency, deposit, complaint). Никаких пояснений, лишних слов или знаков препинания, кроме самой категории.

    Примеры:

    Запрос: "Какой баланс на моей дебетовой карте?"
    Ответ: balance

    Запрос: "Покажите последние 5 операций по счёту"
    Ответ: history

    Запрос: "Переведите 1500 рублей на номер +7 912 345-67-89"
    Ответ: payments

    Запрос: "Хочу взять кредит наличными 300 000 рублей"
    Ответ: credit

    Запрос: "Заблокируйте мою карту, я её потерял"
    Ответ: cards

    Запрос: "Какой курс евро на завтра?"
    Ответ: currency

    Запрос: "Откройте мне вклад под высокий процент"
    Ответ: deposit

    Запрос: "Это возмутительно! Сняли комиссию за смс-информирование дважды"
    Ответ: complaint
    """

    query = f"""
    Теперь классифицируй следующий запрос пользователя. Напиши только одну категорию.
    Запрос: {text}
    Ответ:
    """

    prompt = {
        "modelUri": f"gpt://{CATALOG_ID}/yandexgpt-5-lite",
        "completionOptions": {
            "stream": False,
            "temperature": 0.8,
            "maxTokens": "2000"
        },
        "messages": [
            {
                "role": "system",
                "text": system_prompt
            },
            {
                "role": "user",
                "text": query
            },

        ]
    }


    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Api-Key {API_KEY}"
    }

    start = time.time()

    response = requests.post(url, headers=headers, json=prompt)

    elapsed = time.time() - start

    if response.status_code != 200:
        raise Exception(f"Ошибка {response.status_code}: {response.text}")

    result = response.json()
    # Извлекаем категорию
    category = result["result"]["alternatives"][0]["message"]["text"].strip().lower()
    # На случай, если модель возвращает более 1 слова
    if len(category) > 1:
        category = category.split("\n")[0].split(' ')[0]
    # Извлекаем сведения о токенах
    usage = result.get("result", {}).get("usage", {})
    tokens_in = usage.get("inputTextTokens", 0)
    tokens_out = usage.get("completionTokens", 0)

    return category, elapsed, tokens_in, tokens_out

## 3. Proposed (TF-IDF + LogReg)

Для классификации запросов обучаем логистическую регрессию на простых TF-IDF векторах.

In [86]:
# Обучаем TF-IDF
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=1000)
X_train = vectorizer.fit_transform(train_df["text"])
y_train = train_df["category"]

# Обучаем логистическую регрессию
clf = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42)
clf.fit(X_train, y_train)

# Предсказание на тестовой выборке
X_test = vectorizer.transform(test_df["text"])
y_pred_ml = clf.predict(X_test)
# Отчет с метриками качества классификации
print(classification_report(test_df["category"], y_pred_ml))

# Функция для классификации одного запроса
def classify_with_ml(text):
    vec = vectorizer.transform([text])
    return clf.predict(vec)[0]

              precision    recall  f1-score   support

     balance       0.75      1.00      0.86         3
       cards       1.00      0.67      0.80         3
   complaint       1.00      0.67      0.80         3
      credit       0.75      1.00      0.86         3
    currency       1.00      1.00      1.00         3
     deposit       1.00      0.67      0.80         3
     history       0.40      0.67      0.50         3
    payments       1.00      0.67      0.80         3

    accuracy                           0.79        24
   macro avg       0.86      0.79      0.80        24
weighted avg       0.86      0.79      0.80        24



## 4. Сравнение решений

Сопоставляем baseline и proposed решения на тестовом сэмпле из 10 запросов по точности предсказаний, времени выполнения запросов и количеству входных/выходных токенов (для LLM).

In [71]:
# Возьмем 10 примеров из тестовой выборки
comparison_df = test_df.sample(10, random_state=42)

baseline_results = []
baseline_times = []
baseline_token_in = []
baseline_token_out = []

print("** Запросы к YandexGPT (baseline) **")
for idx, row in comparison_df.iterrows():
    text = row["text"]
    try:
        cat, elapsed, tok_in, tok_out = classify_with_llm(text)
        baseline_results.append(cat)
        baseline_times.append(elapsed)
        baseline_token_in.append(int(tok_in))
        baseline_token_out.append(int(tok_out))
        print(f"Обработано: {text[:40]}... -> {cat} ({elapsed:.2f} с)")
    except Exception as e:
        print(f"Ошибка для '{text[:40]}...' (id={idx}): {e}")
        baseline_results.append("error")
        baseline_times.append(0)
        baseline_token_in.append(0)
        baseline_token_out.append(0)

# Предсказания LogReg
ml_results = [classify_with_ml(text) for text in comparison_df["text"]]

# Добавляем колонки в DataFrame
comparison_df["baseline_pred"] = baseline_results
comparison_df["ml_pred"] = ml_results
comparison_df["true_label"] = comparison_df["category"]
comparison_df["time_llm_sec"] = baseline_times
comparison_df["tokens_in"] = baseline_token_in
comparison_df["tokens_out"] = baseline_token_out

# Функция для вывода результатов
def print_comparison(df):
    print("\n" + "="*120)
    print(f"{'Текст запроса':<50} | {'True':<12} | {'YandexGPT':<12} | {'LogReg (ML)':<12} | {'Время LLM, с':<12} | {'Токены (in/out)':<15}")
    print("="*120)
    for _, row in df.iterrows():
        text_short = row["text"][:47] + "..." if len(row["text"]) > 47 else row["text"]
        tokens_str = f"{row['tokens_in']}/{row['tokens_out']}"
        print(f"{text_short:<50} | {row['true_label']:<12} | {row['baseline_pred']:<12} | {row['ml_pred']:<12} | {row['time_llm_sec']:<12.3f} | {tokens_str:<15}")
    print("="*120)

print_comparison(comparison_df)

# Статистика по baseline
avg_time = np.mean([t for t in baseline_times if t > 0])
avg_tokens_in = np.mean([t for t in baseline_token_in if t > 0])
avg_tokens_out = np.mean([t for t in baseline_token_out if t > 0])
print(f"\nСреднее время ответа YandexGPT: {avg_time:.3f} сек")
print(f"Среднее число входных токенов: {avg_tokens_in:.1f}, выходных: {avg_tokens_out:.1f}")

# Примерная оценка скорости предсказания LogReg
start_ml = time.time()
for text in comparison_df["text"]:
    _ = classify_with_ml(text)
ml_batch_time = time.time() - start_ml
print(f"Время классификации 10 примеров ML: {ml_batch_time:.4f} сек (~{ml_batch_time/10:.4f} сек на запрос)")

** Запросы к YandexGPT (baseline) **
Обработано: Скачать выписку за 2023 год... -> history (1.90 с)
Обработано: Забрать вклад досрочно, не теряя процент... -> deposit (1.00 с)
Обработано: Оплатить штраф ГИБДД через приложение... -> payments (2.60 с)
Обработано: В отделении нахамили, требую разобраться... -> complaint (1.03 с)
Обработано: Активация новой карты... -> cards (2.83 с)
Обработано: Сколько денег доступно к снятию?... -> balance (1.08 с)
Обработано: Курс белорусского рубля к российскому... -> currency (1.09 с)
Обработано: Остаток на счёте в рублях... -> balance (1.65 с)
Обработано: Регулярный платеж за интернет... -> payments (1.14 с)
Обработано: Налог на проценты по вкладам... -> deposit (1.61 с)

Текст запроса                                      | True         | YandexGPT    | LogReg (ML)  | Время LLM, с | Токены (in/out)
Скачать выписку за 2023 год                        | history      | history      | history      | 1.904        | 646/2          
Забрать вклад досрочно, н

## 5. Hybrid (ML + LLM)

Идея гибридного решения: логистическая регрессия выдает вероятности для каждого класса, если максимальная вероятность выше порога (например, 0.7), используем ответ модели; если ниже - запрос сложный/неочевидный, передаем его в LLM.

In [87]:
CONFIDENCE_THRESHOLD = 0.7

def hybrid_classify(text, vectorizer, clf, llm_func, true_label=None):
    """
    Возвращает словарь с результатами гибридной классификации.
    Параметры:
        text - текст запроса
        vectorizer, clf - обученные TF-IDF и логистическая регрессия
        llm_func - функция вызова LLM (должна возвращать категорию, время, токены)
        true_label - истинная категория (опционально, для тестов)
    Возвращает:
        dict с полями:
            final_category - итоговая категория (ML или LLM)
            source - 'ML' или 'LLM'
            ml_confidence - максимальная вероятность от LogReg
            ml_prediction - предсказание ML (всегда)
            llm_prediction - предсказание LLM (если вызывалась, иначе None)
            llm_time_sec - время вызова LLM (0 для ML)
            llm_tokens_in, llm_tokens_out - токены (0 для ML)
            true_label - истинная метка (если передана)
    """
    vec = vectorizer.transform([text])
    probs = clf.predict_proba(vec)[0]
    max_prob = max(probs)
    ml_pred = clf.classes_[probs.argmax()]

    result = {
        'ml_confidence': max_prob,
        'ml_prediction': ml_pred,
        'true_label': true_label,
        'llm_prediction': None,
        'llm_time_sec': 0.0,
        'llm_tokens_in': 0,
        'llm_tokens_out': 0
    }

    if max_prob >= CONFIDENCE_THRESHOLD:
        result['final_category'] = ml_pred
        result['source'] = 'ML'
        return result
    else:
        cat, elapsed, tok_in, tok_out = llm_func(text)
        result['final_category'] = cat
        result['source'] = 'LLM'
        result['llm_prediction'] = cat
        result['llm_time_sec'] = elapsed
        result['llm_tokens_in'] = tok_in
        result['llm_tokens_out'] = tok_out
        return result

# Тест на тех же 10 примерах
print("** Гибридная классификация **\n")

hybrid_records = []
for idx, row in comparison_df.iterrows():
    text = row['text']
    true_cat = row['category']
    res = hybrid_classify(text, vectorizer, clf, classify_with_llm, true_label=true_cat)
    hybrid_records.append(res)

# Печать таблицы с результатами
print(f"{'Текст':<42} | {'True':<10} | {'ML pred':<10} | {'ML conf':<8} | {'LLM pred':<10} | {'Source':<6} | {'Final':<10}")
print("-" * 115)
for rec, text in zip(hybrid_records, comparison_df['text']):
    short = text[:39] + "..." if len(text) > 39 else text
    ml_conf = f"{rec['ml_confidence']:.2f}"
    llm_pred = rec['llm_prediction'] if rec['llm_prediction'] else "-"
    src = rec['source']
    final = rec['final_category']
    true = rec['true_label']
    print(f"{short:<42} | {true:<10} | {rec['ml_prediction']:<10} | {ml_conf:<8} | {llm_pred:<10} | {src:<6} | {final:<10}")

** Гибридная классификация **

Текст                                      | True       | ML pred    | ML conf  | LLM pred   | Source | Final     
-------------------------------------------------------------------------------------------------------------------
Скачать выписку за 2023 год                | history    | history    | 0.15     | history    | LLM    | history   
Забрать вклад досрочно, не теряя процен... | deposit    | deposit    | 0.22     | deposit    | LLM    | deposit   
Оплатить штраф ГИБДД через приложение      | payments   | payments   | 0.18     | payments   | LLM    | payments  
В отделении нахамили, требую разобратьс... | complaint  | complaint  | 0.13     | complaint  | LLM    | complaint 
Активация новой карты                      | cards      | cards      | 0.25     | cards      | LLM    | cards     
Сколько денег доступно к снятию?           | balance    | balance    | 0.19     | balance    | LLM    | balance   
Курс белорусского рубля к российскому      | cur

Мы видим, что модель логистической регрессии не уверена в метках - вероятности находятся около 0.2, поэтому предложенное гибридное решение не экономит ресурсы (все запросы передаются LLM). Наиболее вероятная причина - недостаточно обучающих данных. Ожидается, что при обучении модели на реальном датасете большего размера и при подборе порога уверенности гибридный подход будет наиболее оптимальным по соотношению точности и скорости в сравнении с baseline (только LLM) и proposed (только ML).